In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import h5py
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.data import besInferenceDatapoints

## Static synthetic data

In [ ]:
filepath="/home/molnarbalazs/data/BES_ML_modelling/static synthetic data/Na_beam/"
filename="Na_statdataset.hdf5"
df=pd.read_hdf(os.path.join(filepath, filename), "df")

In [ ]:
df

In [ ]:
type(df['T shape[eV]'].iloc[0])

In [ ]:
df['T_key'] = df['T shape[eV]'].apply(tuple)
unique_T = df['T_key'].unique()
groups = dict(tuple(df.groupby('T_key')))
for ind,T in enumerate(unique_T):
    emission=[]
    [emission.append(i) for i in groups[T]['Emission 3p-->3s'].values]
    emission=np.array(emission)
    density=[]
    [density.append(i) for i in groups[T]['Density Shape[$1/m^3$]'].values]
    density=np.array(density)
    grid=np.linspace(0,0.4,density.shape[1])
    # get the species from the filename by going to the first underscore
    species=filename.split('_')[0]
    temperature=groups[T]['T shape[eV]'].values[0]
    # check if "temperature" is a string, if so, convert it to a float array, e.g. '[45, 34, 21]' -> np.array([45, 34, 21])
    if isinstance(temperature, str):
        temperature = np.fromstring(temperature.strip().strip("[]"), sep=",", dtype=np.float32)
    ID="dss"+str(ind)
    verbose="static synthetic data on a grid with 1mm resolution"
    tags=[str(i) for i in range(density.shape[0])]
    test_data=besInferenceDatapoints(grid=grid,species=species,ID=ID,temperature=temperature,verbose=verbose)
    test_data.add_datapoints_bulk(density, emission, tags)
    print("Created besInferenceDatapoints object with ID:", ID, "species:", species, 
              "temperature shape:", temperature.shape, "grid shape:", grid.shape, "density shape:", density.shape, "emission shape:", emission.shape)  
    

In [ ]:
density.shape

In [ ]:
species

In [ ]:
plt.plot(test_data.grid,test_data.densities[0])
test_data.ID

### grid: 1mm-es lepeskozzel

In [ ]:
df.columns

In [ ]:
df.head()

## Fluctuation data

In [ ]:
filepath="/home/molnarbalazs/data/BES_ML_modelling/synthetic data with fluctuations/Li_beam/"
filenames=[i for i in os.listdir(filepath) if i.endswith(".hdf5")]
df=pd.DataFrame()
for filename in filenames:
    df=pd.concat([df, pd.read_hdf(os.path.join(filepath, filename), "df")])

In [ ]:
len(df)

In [ ]:
df.head()

In [ ]:
[print(type(df['T shape[eV]'].iloc[i])) for i in range(len(df)) if isinstance(df['T shape[eV]'].iloc[i], np.ndarray) == False]

In [ ]:
df['T_key'] = df['T shape[eV]'].apply(lambda x: tuple(x) if isinstance(x, np.ndarray) else (x,))
unique_T = df['T_key'].unique()
groups = dict(tuple(df.groupby('T_key')))
for ind,T in enumerate(unique_T):
    emission=[]
    [emission.append(i) for i in groups[T]['Emission 3p-->3s'].values]
    emission=np.array(emission)
    density=[]
    [density.append(i) for i in groups[T]['Density Shape[$1/m^3$]'].values]
    density=np.array(density)
    if len(density.shape) < 2:
        # skip this iteration if density is 1D
        continue
    print("Density shape:", density.shape)
    grid=np.linspace(0,0.4,density.shape[1])
    species=filename.split('_')[0]
    temperature=groups[T]['T shape[eV]'].values[0]
    ID="dsf"+str(ind)
    verbose="fluctuation synthetic data"
    tags=[str(i) for i in range(density.shape[0])]
    test_data=besInferenceDatapoints(grid=grid,species=species,ID=ID,temperature=temperature,verbose=verbose)
    test_data.add_datapoints_bulk(density, emission, tags)
    print("Created besInferenceDatapoints object with ID:", ID, "species:", species, 
              "temperature shape:", temperature.shape, "grid shape:", grid.shape, "density shape:", density.shape, "emission shape:", emission.shape)

In [ ]:
plt.plot(test_data.grid,test_data.densities.T)
test_data.ID

## Hesel data

In [ ]:
filepath="/home/molnarbalazs/data/BES_ML_modelling/HESEL data/"
filenames=os.listdir(filepath)
fast_files=[i for i in filenames if (i.endswith(".h5") and "fast" in i)]
slow_files=[i for i in filenames if (i.endswith(".h5") and "fast" not in i)]

In [ ]:
for ind,filename in enumerate(slow_files):
    with h5py.File(os.path.join(filepath, filename), 'r') as f:
        if ind==0:
            energy = f['Beam energy [keV]'][()]
            species=f['Beam type'][()].decode("utf-8")
            grid=f['Grid [m]'][()]
            density=np.empty((0,grid.shape[0]))
            emission=np.empty((0,grid.shape[0]))
            temperature=np.empty((0,grid.shape[0]))     
        else:
            if f['Beam energy [keV]'][()] != energy:
                raise ValueError(f"Energy mismatch: {f['Beam energy [keV]'][()]} keV != {energy} keV")
            if f['Beam type'][()].decode("utf-8") != species:
                raise ValueError(f"Species mismatch: {f['Beam type'][()].decode('utf-8')} != {species}")
            if np.array_equal(f['Grid [m]'][()], grid) == False:
                raise ValueError(f"Grid mismatch: {f['Grid [m]'][()]} != {grid}")
            energy = f['Beam energy [keV]'][()]
            species=f['Beam type'][()].decode("utf-8")
            grid=f['Grid [m]'][()]
        density=np.vstack([density, f['Density [m-3]'][()]])
        emission=np.vstack([emission, f['Population [-]'][()]])
        temperature=np.vstack([temperature, f['Temperature profiles [eV]'][()]])
temperature=temperature.mean(axis=0)
print(f"Loaded slow HESEL data with energy {energy} keV, species {species}, grid shape {grid.shape}, density shape {density.shape}, emission shape {emission.shape}, temperature shape {temperature.shape}")
ID="ash"+str(ind)
verbose="slow HESEL synthetic data with average temperature profile"
tags=[str(i) for i in range(density.shape[0])]
test_data=besInferenceDatapoints(grid=grid,energy=energy,species=species,ID=ID,temperature=temperature,verbose=verbose)
test_data.add_datapoints_bulk(density, emission, tags)

In [ ]:
plt.plot(grid,density[::100].T)

In [ ]:
plt.plot(grid,emission[100])

In [ ]:
for ind,filename in enumerate(fast_files):
    with h5py.File(os.path.join(filepath, filename), 'r') as f:
        if ind==0:
            energy = f['Beam energy [keV]'][()]
            species=f['Beam type'][()].decode("utf-8")
            grid=f['Grid [m]'][()]
            density=np.empty((0,grid.shape[0]))
            emission=np.empty((0,grid.shape[0]))
            temperature=np.empty((0,grid.shape[0]))     
        else:
            if f['Beam energy [keV]'][()] != energy:
                raise ValueError(f"Energy mismatch: {f['Beam energy [keV]'][()]} keV != {energy} keV")
            if f['Beam type'][()].decode("utf-8") != species:
                raise ValueError(f"Species mismatch: {f['Beam type'][()].decode('utf-8')} != {species}")
            if np.array_equal(f['Grid [m]'][()], grid) == False:
                raise ValueError(f"Grid mismatch: {f['Grid [m]'][()]} != {grid}")
            energy = f['Beam energy [keV]'][()]
            species=f['Beam type'][()].decode("utf-8")
            grid=f['Grid [m]'][()]
        density=np.vstack([density, f['Density [m-3]'][()]])
        emission=np.vstack([emission, f['Population [-]'][()]])
        temperature=np.vstack([temperature, f['Temperature profiles [eV]'][()]])
temperature=temperature.mean(axis=0)
print(f"Loaded fast HESEL data with energy {energy} keV, species {species}, grid shape {grid.shape}, density shape {density.shape}, emission shape {emission.shape}, temperature shape {temperature.shape}")
ID="ash"+str(ind)
verbose="fast HESEL synthetic data with average temperature profile"
tags=[str(i) for i in range(density.shape[0])]
test_data=besInferenceDatapoints(grid=grid,energy=energy,species=species,ID=ID,temperature=temperature,verbose=verbose)
test_data.add_datapoints_bulk(density, emission, tags)

In [ ]:
plt.plot(grid,density[100])

In [ ]:
plt.plot(grid,emission[100])

## ASDEX data

In [ ]:
with h5py.File("/home/molnarbalazs/data/BES_ML_modelling/ASDEX data/dataset.h5", 'r') as f:
    print(f.keys())
    print(f['s40701'].keys())
    print(f['s42425'].keys())